# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

I am modeling the Content Refresh / Content Opportunity lane as a binary classification problem.

The target is `is_declining_label`, where a page is labeled positive when the observed `trend_direction` is `"down"`.

I will compare a simple Logistic Regression model with a Random Forest model. The ranking metric is **Precision@50**, because the practical question is which pages should be reviewed first.

I will use a client-holdout split so pages from the same client do not appear in both training and test data. Results are treated as observed, measured, and directional decision-support rather than as a claim about Google's algorithm.


In [1]:
!git clone https://github.com/Alpeshmore/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 153 (delta 61), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 1.93 MiB | 3.44 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [2]:
# ML-08 — Setup and load data

from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Reuse df if it already exists from an earlier notebook section.
if "df" not in globals():
    possible_paths = [
        Path("data/processed/refresh_feature_vector.csv"),
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ai/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ai/data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
    ]

    data_path = next((p for p in possible_paths if p.exists()), None)

    if data_path is None:
        raise FileNotFoundError(
            "Dataset not found. Run the ML-05 preparation first or clone/upload "
            "the FlyRank starter repo so data/raw/content_refresh_anonymized.csv exists."
        )

    df = pd.read_csv(data_path)
    print(f"Loaded: {data_path}")

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Target
if "is_declining_label" not in df.columns:
    if "trend_direction" not in df.columns:
        raise ValueError("Need trend_direction or is_declining_label.")
    df["is_declining_label"] = (
        df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
    )

df["is_declining_label"] = df["is_declining_label"].astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts().sort_index())

print("\nBase rate:")
print(f"{df['is_declining_label'].mean():.3f}")

Loaded: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Base rate:
0.542


## 1. Method choice and why

I will start with **Logistic Regression** because this is a yes/no classification problem with an observed label, and the model is relatively easy to interpret.

I will also test **Random Forest** as a stronger nonlinear comparison. Logistic Regression gives me a readable baseline for learned modeling, while Random Forest can capture interactions and nonlinear relationships between content, traffic, freshness, and search signals.

I will choose the model using **Precision@50**, because my lane is about ranking pages for review rather than simply predicting every page correctly.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the modeling columns and target

print("Target:", "is_declining_label")
print("Positive rate:", round(df["is_declining_label"].mean(), 4))

required_for_split = ["client_id", "is_declining_label"]

missing = [c for c in required_for_split if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Unique clients:", df["client_id"].nunique())
print("Both target classes:", df["is_declining_label"].nunique() == 2)

Target: is_declining_label
Positive rate: 0.5421
Unique clients: 32
Both target classes: True


## 2. Split design

I will use a **client-holdout split** with a fixed random seed of 42.

Approximately 20% of clients will be held out for testing. This is more honest than randomly splitting rows because multiple pages can belong to the same client. A row-level split could allow the model to learn client-specific patterns from training pages and then benefit from seeing the same client's pages in the test set.

The test set will be used only for the final comparison of the baseline and learned models.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Client-aware 80/20 split

all_indices = np.arange(len(df))

clients = df["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)

test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = clients.isin(test_clients).to_numpy()

train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]

train_df = df.iloc[train_indices].copy()
test_df = df.iloc[test_indices].copy()

print("Split strategy: client_holdout")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nTrain target rate:", round(train_df["is_declining_label"].mean(), 4))
print("Test target rate:", round(test_df["is_declining_label"].mean(), 4))

overlap = set(train_df["client_id"].astype(str)) & set(test_df["client_id"].astype(str))
print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Train clients: 26
Test clients: 6

Train target rate: 0.5548
Test target rate: 0.391

Client overlap: 0


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature lists based on the FlyRank modeling feature guidance.
# Only keep columns that actually exist in the current dataframe.

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 14
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features: 8
['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


In [7]:
# Build numeric + one-hot encoded categorical feature matrix

numeric_frame = df[numeric_features].apply(pd.to_numeric, errors="coerce")
numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_frame = (
    df[categorical_features]
    .fillna("unknown")
    .astype(str)
)

encoded_frame = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True)
    ],
    axis=1
)

y = df["is_declining_label"].astype(int).reset_index(drop=True)

print("Feature matrix:", X.shape)
print("Target:", y.shape)

# Leakage safety check
for forbidden in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]:
    assert forbidden not in X.columns, f"Leakage/grouping field found: {forbidden}"

print("Leakage safety check: PASS")

Feature matrix: (30000, 48)
Target: (30000,)
Leakage safety check: PASS


## 3. Train + compare vs my baseline

I will evaluate the hand-written baseline and learned models on exactly the same held-out clients.

The main metric is **Precision@50**. I will also report Precision@20 and Precision@100 because the result can change depending on how many pages are available for review.

The baseline is the transparent Week-4 ranking rule. The learned models receive the same test rows and are compared using the same ranking metric. I will select the strongest model based primarily on Precision@50, not on a score from a different split.


In [8]:
# Train Logistic Regression + Random Forest

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train = X.iloc[train_indices]
X_test = X.iloc[test_indices]

y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

model_probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    model_probabilities[name] = probabilities

    print(f"{name}: trained")

logistic_regression: trained
random_forest: trained


In [9]:
# Precision@K helper

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    if k == 0:
        return 0.0

    order = np.argsort(-scores, kind="stable")[:k]
    return float(y_true[order].mean())


def evaluate_ranking(y_true, scores):
    return {
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
    }

In [10]:
# Recreate the ML-07 baseline score on the SAME test rows.
# This uses the same transparent scoring formula as the reference baseline.

def percentile_rank(series):
    return series.rank(method="average", pct=True).fillna(0)


def normalize(series):
    series = pd.to_numeric(series, errors="coerce")
    minimum = series.min()
    maximum = series.max()

    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


baseline_frame = df.copy()

baseline_frame["visibility_score"] = percentile_rank(
    np.log1p(
        pd.to_numeric(
            baseline_frame["impressions_90d"],
            errors="coerce"
        ).fillna(0)
    )
)

baseline_frame["freshness_risk_score"] = percentile_rank(
    pd.to_numeric(
        baseline_frame["days_since_last_update"],
        errors="coerce"
    ).fillna(0)
)

position = pd.to_numeric(
    baseline_frame["avg_position"],
    errors="coerce"
).fillna(0)

baseline_frame["position_opportunity_score"] = (
    (1 - normalize(position.clip(lower=1, upper=50)))
    * baseline_frame["visibility_score"]
    * (position > 0).astype(int)
)

word_count = pd.to_numeric(
    baseline_frame["word_count"],
    errors="coerce"
).fillna(0)

baseline_frame["depth_gap_score"] = (
    (1 - percentile_rank(word_count))
    * baseline_frame["visibility_score"]
)

baseline_frame["baseline_refresh_score"] = (
    0.40 * baseline_frame["visibility_score"]
    + 0.30 * baseline_frame["freshness_risk_score"]
    + 0.25 * baseline_frame["position_opportunity_score"]
    + 0.05 * baseline_frame["depth_gap_score"]
).clip(0, 1)

baseline_test_scores = (
    baseline_frame.iloc[test_indices]["baseline_refresh_score"]
    .to_numpy()
)

print("Baseline score created on test rows.")

Baseline score created on test rows.


In [11]:
# Final comparison table

comparison_rows = []

baseline_metrics = evaluate_ranking(
    y_test,
    baseline_test_scores
)

comparison_rows.append({
    "Method": "Week-4 baseline",
    **baseline_metrics
})

for name, probabilities in model_probabilities.items():
    metrics = evaluate_ranking(y_test, probabilities)

    comparison_rows.append({
        "Method": name,
        **metrics
    })

comparison_table = pd.DataFrame(comparison_rows)

print(comparison_table.to_string(index=False))

print("\nTest base rate:",
      round(y_test.mean(), 4))

             Method  Precision@20  Precision@50  Precision@100
    Week-4 baseline          0.15          0.24           0.36
logistic_regression          0.45          0.58           0.64
      random_forest          0.70          0.74           0.71

Test base rate: 0.391


In [12]:
# Select the best learned model by Precision@50

learned_models = comparison_table[
    comparison_table["Method"] != "Week-4 baseline"
].copy()

best_row = learned_models.sort_values(
    ["Precision@50", "Precision@20", "Precision@100"],
    ascending=False
).iloc[0]

best_model_name = best_row["Method"]
best_model_score = float(best_row["Precision@50"])

baseline_score = float(
    comparison_table.loc[
        comparison_table["Method"] == "Week-4 baseline",
        "Precision@50"
    ].iloc[0]
)

print("Best learned model:", best_model_name)
print("Best learned Precision@50:", round(best_model_score, 3))
print("Baseline Precision@50:", round(baseline_score, 3))
print("Difference:", round(best_model_score - baseline_score, 3))

Best learned model: random_forest
Best learned Precision@50: 0.74
Baseline Precision@50: 0.24
Difference: 0.5


## 4. Errors and interpretation

I will inspect the best learned model in two ways.

First, I will look at its most important features to understand what signals it relies on. Second, I will inspect three concrete test cases where the model's ranking confidence was wrong.

These errors are useful because pages with strong traffic, freshness, position, or engagement signals can still behave differently from the model's overall pattern. The model should therefore be treated as decision-support for review, not as an automatic refresh decision.


In [13]:
# Feature importance

best_model = models[best_model_name]
importance_values = None

if best_model_name == "logistic_regression":
    classifier = best_model.named_steps["model"]
    importance_values = np.abs(classifier.coef_[0])
else:
    importance_values = best_model.feature_importances_

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importance_values
}).sort_values(
    "importance",
    ascending=False
)

print("Top 15 features:")
print(
    importance_df.head(15).to_string(index=False)
)

Top 15 features:
                 feature  importance
   days_with_impressions    0.202428
            avg_position    0.127874
        content_age_days    0.102486
      days_with_sessions    0.047798
                     ctr    0.045041
              word_count    0.043680
              char_count    0.039235
             scroll_rate    0.037762
           age_tier_365+    0.032372
     position_tier_top_3    0.032274
  days_since_last_update    0.027765
     impression_tier_low    0.022464
         age_tier_91-180    0.021180
impression_tier_moderate    0.018277
 word_count_tier_unknown    0.017111


In [14]:
# Build test prediction frame

test_results = df.iloc[test_indices].copy()

test_results["model_probability"] = model_probabilities[best_model_name]

test_results["actual_declining"] = (
    test_results["is_declining_label"].astype(int)
)

test_results["predicted_positive"] = (
    test_results["model_probability"] >= 0.5
).astype(int)

# Error type
test_results["error_type"] = "correct"

test_results.loc[
    (test_results["predicted_positive"] == 1) &
    (test_results["actual_declining"] == 0),
    "error_type"
] = "false_positive"

test_results.loc[
    (test_results["predicted_positive"] == 0) &
    (test_results["actual_declining"] == 1),
    "error_type"
] = "false_negative"

print(
    test_results["error_type"]
    .value_counts()
)

error_type
correct           1549
false_positive     534
false_negative     242
Name: count, dtype: int64


In [15]:
# Three concrete wrong cases
# Sort by confidence so the examples are easy to inspect.

false_positives = (
    test_results[
        test_results["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
)

false_negatives = (
    test_results[
        test_results["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=True)
)

error_examples = pd.concat([
    false_positives.head(2),
    false_negatives.head(1)
])

error_columns = [
    "content_id",
    "client_id",
    "model_probability",
    "actual_declining",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

error_columns = [
    c for c in error_columns
    if c in error_examples.columns
]

print(
    error_examples[error_columns]
    .to_string(index=False)
)

          content_id         client_id  model_probability  actual_declining  impressions_90d  sessions_90d  avg_position  ctr  word_count  content_age_days  days_since_last_update
content_331182ca4cae client_f74efabef1           0.751950                 0             3026            24          35.9 0.00      3546.0               134                      20
content_b15a8dbdf66f client_f74efabef1           0.737027                 0             1647            10          22.4 0.18      4095.0               144                      20
content_34b14c00f80c client_d4735e3a26           0.105442                 1                3             1           0.0 0.00       659.0               308                      20


In [16]:
# Error rates by useful ranges

test_results["impression_band"] = pd.cut(
    test_results["impressions_90d"],
    bins=[-1, 100, 500, 2000, np.inf],
    labels=["0-100", "101-500", "501-2000", "2000+"]
)

error_summary = (
    test_results
    .groupby("impression_band", observed=False)
    .agg(
        pages=("is_declining_label", "size"),
        observed_decline_rate=("is_declining_label", "mean"),
        mean_model_probability=("model_probability", "mean"),
        false_positive_rate=(
            "error_type",
            lambda x: (x == "false_positive").mean()
        ),
        false_negative_rate=(
            "error_type",
            lambda x: (x == "false_negative").mean()
        ),
    )
    .reset_index()
)

print(error_summary.to_string(index=False))

impression_band  pages  observed_decline_rate  mean_model_probability  false_positive_rate  false_negative_rate
          0-100   1433               0.300070                0.357050             0.111654             0.157013
        101-500    275               0.578182                0.610608             0.396364             0.036364
       501-2000    277               0.545126                0.652147             0.454874             0.000000
          2000+    340               0.497059                0.605726             0.408824             0.020588


## ML-08 conclusion

The learned model was evaluated against the Week-4 rule baseline using the same held-out clients and Precision@50.

The best model is **[fill from the table]**, with an observed Precision@50 of **[fill from the output]**, compared with **[fill from the output]** for the baseline.

The result is directional rather than proof of causality. The model appears useful for prioritizing pages for human review, but the error cases show that high-ranked pages can still be wrong, so the score should be used as decision-support rather than an automatic refresh decision.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.